# DistilBERT fine-tuning for the model comparison (feature 02)

Run this on **Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU). About 15–25 min.

It trains and tests on **exactly the same rows** as the classical models, using the
`results/split_indices.json` file written by `python experiments/run_dataset_profile.py`.

**Upload these two files** when asked (cell 2):
1. `data/fake reviews dataset.csv`
2. `results/split_indices.json`

**Download** the two output files at the end and put them in the repo at
`results/predictions/distilbert.csv` and `results/predictions/distilbert_meta.json`,
then run `python experiments/run_model_comparison.py --only distilbert`
(or the full script) to merge BERT into the leaderboard.

No GPU? Set `TRAIN_SUBSET = 4000` below and say "DistilBERT trained on a 4k subset" in the report.

In [ ]:
!pip -q install transformers datasets accelerate

In [ ]:
# --- upload inputs ---
import os
try:
    from google.colab import files
    uploaded = files.upload()   # pick: 'fake reviews dataset.csv' and 'split_indices.json'
except ImportError:
    print("Not on Colab: put both files in the working directory.")

DATA_CSV = "fake reviews dataset.csv"
SPLIT_JSON = "split_indices.json"
assert os.path.exists(DATA_CSV) and os.path.exists(SPLIT_JSON)

In [ ]:
# --- settings (keep identical across team runs) ---
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
EPOCHS = 2
LR = 2e-5
BATCH = 32
SEED = 42
TRAIN_SUBSET = None   # e.g. 4000 for a CPU-only run

In [ ]:
# --- rebuild the exact rows the repo uses ---
# Mirrors src/kaggle_adapter.py (review_id = 1..n in file order, genuine = label 'OR')
# and src/preprocessing.clean_text (strip URLs, collapse whitespace).
import json, re
import pandas as pd

raw = pd.read_csv(DATA_CSV)
raw.columns = [c.strip() for c in raw.columns]
df = pd.DataFrame({
    "review_id": range(1, len(raw) + 1),
    "text": raw["text_"].astype(str),
    "label": (raw["label"] == "OR").astype(int),
})

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", " ", str(text))
    return re.sub(r"\s+", " ", text).strip()

df["text"] = df["text"].apply(clean_text)

split = json.load(open(SPLIT_JSON))
train_df = df.set_index("review_id").loc[split["train"]].reset_index()
test_df = df.set_index("review_id").loc[split["test"]].reset_index()
if TRAIN_SUBSET:
    train_df = train_df.sample(TRAIN_SUBSET, random_state=SEED)
print(len(train_df), "train /", len(test_df), "test")
print(train_df["label"].value_counts().to_dict())

In [ ]:
# --- tokenize ---
import numpy as np, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize, batched=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# --- fine-tune ---
import time
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args = TrainingArguments(
    output_dir="bert_out",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    weight_decay=0.01,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, tokenizer=tokenizer)

start = time.perf_counter()
trainer.train()
train_seconds = time.perf_counter() - start
print(f"train: {train_seconds/60:.1f} min")

In [ ]:
# --- predict on the shared test split ---
start = time.perf_counter()
logits = trainer.predict(test_ds).predictions
predict_seconds = time.perf_counter() - start

proba = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
y_pred = (proba >= 0.5).astype(int)
y_true = test_df["label"].to_numpy()
print("accuracy:", (y_pred == y_true).mean().round(4))

In [ ]:
# --- save outputs in the same format as the classical models ---
pd.DataFrame({"review_id": test_df["review_id"], "y_true": y_true,
              "y_pred": y_pred, "score": proba}).to_csv("distilbert.csv", index=False)
meta = {
    "model_name": MODEL_NAME, "max_length": MAX_LENGTH, "epochs": EPOCHS,
    "learning_rate": LR, "batch_size": BATCH, "seed": SEED,
    "train_subset": TRAIN_SUBSET, "n_train": len(train_df), "n_test": len(test_df),
    "train_seconds": round(train_seconds, 1), "predict_seconds": round(predict_seconds, 1),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
json.dump(meta, open("distilbert_meta.json", "w"), indent=2)
try:
    files.download("distilbert.csv"); files.download("distilbert_meta.json")
except NameError:
    print("Saved distilbert.csv and distilbert_meta.json")

Optional: to keep the fine-tuned weights (≈260 MB), save them to Google Drive with
`trainer.save_model('/content/drive/MyDrive/distilbert_reviews')`. **Never commit them to git.**